# V19 Path 3b — MFN + IR-ResNet50 + CCM-on-MFN Multi-Backbone Training (Colab L4)

**Plan reference:** `~/.claude/plans/you-are-helping-implement-witty-river.md` §V19 + §V19.10 (Scenario A).

**Goal:** train 3 backbones (MobileFaceNet, IR-ResNet50, CCM-on-MFN) on **Tongji + IITD + XJTU-UP-Huawei** (3 datasets × 3 backbones = **9 trainings total**) under the unified Family-A recipe.

**Pre-requisites:**
- Colab runtime: GPU L4 (Runtime → Change runtime type → L4 GPU)
- Tongji, IITD, and XJTU-UP-Huawei ROI datasets on Google Drive
- Repo at `https://github.com/eunsu0325/Secure.git` at commit including V19.10 changes

**Recipe (V19.2 unified):** SGD lr=0.01 mom=0.9 wd=1e-4, batch=128 (P=32×K=4), 50 epochs cosine + 1ep warmup, ArcFace m=0.5 s=48, 112×112 3-channel RGB-replicated input, embedding_dim=512.

**Estimated time on L4 (per training):**
- MFN (~1M params): ~5-10 min
- IR-ResNet50 (~25M params): ~15-25 min
- CCM-on-MFN (MFN + small CCM adapter): ~10-15 min

Total for 3 datasets × 3 backbones ≈ 2.5–3.5 hours.

## 0. Sanity checks

In [ ]:
!nvidia-smi | head -20
import torch
print(f"\ntorch: {torch.__version__}")
print(f"cuda: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cd /content && (rm -rf Secure && git clone https://github.com/eunsu0325/Secure.git)
%cd /content/Secure
!git log --oneline -5

## 2. Path setup — **EDIT BEFORE RUNNING**
Set Drive paths for each dataset's ROI root.

In [ ]:
# === EDIT THESE to match your Drive paths ===
TONGJI_DRIVE_ROOT = '/content/drive/MyDrive/Tongji_extracted/ROI'  # EDIT
IITD_DRIVE_ROOT   = '/content/drive/MyDrive/IITD_Palmprint_V1/Segmented'  # EDIT
# XJTU-UP-Huawei: either point at extracted huawei/ dir, or at a Drive zip to extract
XJTU_UP_HUAWEI_DRIVE_ROOT = '/content/drive/MyDrive/Huawei/huawei'  # EDIT (after extraction)
XJTU_UP_HUAWEI_DRIVE_ZIP  = '/content/drive/MyDrive/Huawei.zip'    # OPTIONAL: zip to extract

TONGJI_LOCAL_PREFIX  = '/Users/kimeunsu/Tongji_extracted/ROI'
IITD_LOCAL_PREFIX    = '/Users/kimeunsu/IITD Palmprint V1/Segmented'

import os
assert os.path.isdir(TONGJI_DRIVE_ROOT), f'Tongji root missing: {TONGJI_DRIVE_ROOT}'
print(f'Tongji root OK: {TONGJI_DRIVE_ROOT}')
if os.path.isdir(IITD_DRIVE_ROOT):
    print(f'IITD root OK:   {IITD_DRIVE_ROOT}')
else:
    print(f'[note] IITD root not found at {IITD_DRIVE_ROOT}; IITD cells will fail.')
if os.path.isdir(XJTU_UP_HUAWEI_DRIVE_ROOT):
    print(f'XJTU-UP-Huawei root OK: {XJTU_UP_HUAWEI_DRIVE_ROOT}')
elif os.path.isfile(XJTU_UP_HUAWEI_DRIVE_ZIP):
    print(f'[note] XJTU-UP-Huawei zip found at {XJTU_UP_HUAWEI_DRIVE_ZIP}; will extract in section 4')
else:
    print(f'[note] XJTU-UP-Huawei neither extracted dir nor zip found; XJTU-UP cells will fail.')

## 3. Manifest path remap (Tongji, IITD)
Rewrites `image_path` column in pre-existing manifest CSVs from local Mac to Drive.

In [ ]:
import pandas as pd
from pathlib import Path

def remap_manifest(manifest_csv, local_prefix, drive_prefix):
    df = pd.read_csv(manifest_csv)
    n_before = len(df)
    sample_before = df['image_path'].iloc[0]
    df['image_path'] = df['image_path'].str.replace(local_prefix, drive_prefix, regex=False)
    sample_after = df['image_path'].iloc[0]
    backup = manifest_csv.with_suffix('.local_paths.csv')
    if not backup.exists():
        manifest_csv.rename(backup)
    df.to_csv(manifest_csv, index=False)
    print(f'  rows: {n_before}; example: {sample_before} -> {sample_after}')
    assert Path(sample_after).exists(), f'Drive image not found: {sample_after}'

tongji_manifest = Path('/content/Secure/experiments/generated/tongji_full/manifest.csv')
iitd_manifest   = Path('/content/Secure/experiments/generated/iitd_full/manifest.csv')

print('--- Tongji remap ---')
remap_manifest(tongji_manifest, TONGJI_LOCAL_PREFIX, TONGJI_DRIVE_ROOT)
if iitd_manifest.exists() and os.path.isdir(IITD_DRIVE_ROOT):
    print('\n--- IITD remap ---')
    remap_manifest(iitd_manifest, IITD_LOCAL_PREFIX, IITD_DRIVE_ROOT)

## 4. XJTU-UP-Huawei: extract zip (if needed) + build manifest from scratch
Unlike Tongji/IITD, XJTU-UP has no pre-existing manifest — we build one in Colab.

In [ ]:
import os, shutil
from pathlib import Path

# If XJTU-UP-Huawei is still a zip on Drive, extract it to /content/xjtu_up_huawei
if not os.path.isdir(XJTU_UP_HUAWEI_DRIVE_ROOT):
    if not os.path.isfile(XJTU_UP_HUAWEI_DRIVE_ZIP):
        raise FileNotFoundError('Neither extracted dir nor zip found for XJTU-UP-Huawei.')
    extract_root = Path('/content/xjtu_up_huawei_extract')
    extract_root.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {XJTU_UP_HUAWEI_DRIVE_ZIP} ...')
    !unzip -q -o '$XJTU_UP_HUAWEI_DRIVE_ZIP' -d '$extract_root'
    # Find the huawei/ folder inside the extracted dir
    candidates = list(extract_root.rglob('huawei'))
    candidates = [c for c in candidates if c.is_dir() and any(c.iterdir())]
    if not candidates:
        raise FileNotFoundError(f'No huawei/ subdir under {extract_root}')
    XJTU_UP_HUAWEI_ROOT = str(candidates[0])
    print(f'Resolved XJTU-UP-Huawei root: {XJTU_UP_HUAWEI_ROOT}')
else:
    XJTU_UP_HUAWEI_ROOT = XJTU_UP_HUAWEI_DRIVE_ROOT

# Sanity: Flash/ and Nature/ should exist under root
for cond in ('Flash', 'Nature'):
    assert (Path(XJTU_UP_HUAWEI_ROOT) / cond).is_dir(), f'Missing condition subdir: {cond}'
print(f'XJTU-UP-Huawei structure confirmed at {XJTU_UP_HUAWEI_ROOT}')
print(f'  Flash/: {len(list((Path(XJTU_UP_HUAWEI_ROOT)/"Flash").iterdir()))} entries')
print(f'  Nature/: {len(list((Path(XJTU_UP_HUAWEI_ROOT)/"Nature").iterdir()))} entries')

In [ ]:
# Build the XJTU-UP-Huawei manifest with the same seed/K conventions used by
# Tongji/IITD (per plan §V19.10).
%env PYTHONPATH=/content/Secure
%cd /content/Secure
!python -u -m exp1_baselines.datasets.xjtu_up_huawei_manifest \
    --root '$XJTU_UP_HUAWEI_ROOT' \
    --out experiments/generated/xjtu_up_huawei_full \
    --seed 42 --K 3 --enroll_seed 0

## 5. Bulk config update — Colab CUDA num_workers + Tongji MFN paths fix

In [ ]:
import yaml
from pathlib import Path

all_configs = [
    'exp1_baselines/configs/mfn_arcface_tongji_112.yaml',
    'exp1_baselines/configs/ir50_arcface_tongji_112.yaml',
    'exp1_baselines/configs/ccm_mfn_arcface_tongji_112.yaml',
    'exp1_baselines/configs/ir50_arcface_iitd_112.yaml',
    'exp1_baselines/configs/ccm_mfn_arcface_iitd_112.yaml',
    'exp1_baselines/configs/mfn_arcface_xjtu_up_huawei_112.yaml',
    'exp1_baselines/configs/ir50_arcface_xjtu_up_huawei_112.yaml',
    'exp1_baselines/configs/ccm_mfn_arcface_xjtu_up_huawei_112.yaml',
]
for cfg_name in all_configs:
    p = Path('/content/Secure') / cfg_name
    if not p.exists():
        print(f'  [skip] {cfg_name} not found')
        continue
    with open(p) as f: cfg = yaml.safe_load(f)
    cfg['train']['num_workers'] = 2  # Colab CUDA: 2 workers safe
    # Fix stale paths in mfn_tongji config
    if 'mfn_arcface_tongji_112' in cfg_name:
        cfg['dataset']['manifest_dir'] = 'experiments/generated/tongji_full'
        cfg['output']['checkpoint_path'] = 'experiments/generated/tongji_full/mfn_arcface_tongji_scratch_112.pt'
        cfg['output']['log_dir'] = 'experiments/generated/tongji_full/mfn_logs'
    with open(p, 'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)
    print(f'updated {cfg_name}')

## 6. V19.9 smoke test on CCM-on-MFN

In [ ]:
!python -u exp1_baselines/backbones/ccm_mfn.py

## 7. Train all 9 (3 datasets × 3 backbones)

Recommended order: smaller-dataset → larger-dataset, lighter-model → heavier-model.
Each cell can be skipped/restarted independently if anything fails.

In [ ]:
%env PYTHONUNBUFFERED=1
%env PYTHONPATH=/content/Secure
%cd /content/Secure
# === XJTU-UP-Huawei (smallest, ~110 train palms × 20 imgs ≈ 2200 train imgs) ===
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/mfn_arcface_xjtu_up_huawei_112.yaml

In [ ]:
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/ir50_arcface_xjtu_up_huawei_112.yaml

In [ ]:
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/ccm_mfn_arcface_xjtu_up_huawei_112.yaml

In [ ]:
# === IITD (260 train palms × ~5 imgs ≈ 1300 train imgs) ===
from pathlib import Path
iitd_mfn_cfg = Path('/content/Secure/exp1_baselines/configs/mfn_arcface_iitd_112.yaml')
if iitd_mfn_cfg.exists():
    !python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/mfn_arcface_iitd_112.yaml
else:
    print('[iitd mfn config missing; using tongji config with overrides]')
    !python -u -m exp1_baselines.train_arcface \
        --config exp1_baselines/configs/mfn_arcface_tongji_112.yaml \
        --manifest_dir experiments/generated/iitd_full \
        --output experiments/generated/iitd_full/mfn_arcface_iitd_scratch_112.pt

In [ ]:
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/ir50_arcface_iitd_112.yaml

In [ ]:
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/ccm_mfn_arcface_iitd_112.yaml

In [ ]:
# === Tongji (320 train palms × 18 imgs ≈ 5760 train imgs; largest) ===
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/mfn_arcface_tongji_112.yaml

In [ ]:
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/ir50_arcface_tongji_112.yaml

In [ ]:
!python -u -m exp1_baselines.train_arcface --config exp1_baselines/configs/ccm_mfn_arcface_tongji_112.yaml

## 8. Copy outputs to Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/V19_Path3b_ckpts')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

for ds in ['tongji', 'iitd', 'xjtu_up_huawei']:
    src_dir = Path(f'/content/Secure/experiments/generated/{ds}_full')
    dst_dir = DRIVE_OUT / f'{ds}_full'
    dst_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n[{ds}] copying from {src_dir}')
    for arch in ['mfn', 'ir50', 'ccm_mfn']:
        for suffix in (f'arcface_{ds}_scratch_112.pt', f'arcface_{ds}_scratch_112.v17_metadata.json'):
            fname = f'{arch}_{suffix}'
            src = src_dir / fname
            if not src.exists():
                print(f'  [missing] {fname}')
                continue
            dst = dst_dir / fname
            shutil.copy(src, dst)
            sz_mb = dst.stat().st_size / 1e6
            print(f'  {fname} ({sz_mb:.1f} MB)')
    # Copy manifest + metadata + logs
    for extra in ['manifest.csv', 'metadata.json']:
        if (src_dir / extra).exists():
            shutil.copy(src_dir / extra, dst_dir / extra)
    for log_dir in src_dir.glob('*_logs'):
        tl = log_dir / 'train_log.json'
        if tl.exists():
            shutil.copy(tl, dst_dir / f'{log_dir.name}_train_log.json')
print(f'\nDone. Outputs at: {DRIVE_OUT}')

## 9. Done
Download `V19_Path3b_ckpts/` to local at `/Users/kimeunsu/Desktop/Research Notebook/secure/Secure/experiments/generated/`, preserving the `<dataset>_full/` substructure. Then run V17.10 gate locally:
```bash
cd '/Users/kimeunsu/Desktop/Research Notebook/secure/Secure'
bash experiments/run_phase5_v19_gate.sh tongji all
bash experiments/run_phase5_v19_gate.sh iitd all
bash experiments/run_phase5_v19_gate.sh xjtu_up_huawei all
```
Per-(backbone, dataset) gate verdicts will appear in `experiments/generated/<dataset>_full/<arch>_protocols/gate_metadata.json`.